# 01 · Datos, etiquetas y análisis exploratorio

En este notebook:
1. Cargamos los datos de CORDIS (`projects.xlsx`) y la jerarquía EuroSciVoc (`SciVocCodes.xlsx`).
2. Construimos la variable objetivo (nivel 2 de EuroSciVoc) siguiendo la sección 3 del enunciado.
3. Hacemos la partición train/val/test que usará todo el equipo.
4. *(Siguiente paso)* Análisis exploratorio.

In [15]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# En Colab los ficheros subidos están en /content/; en local, en la carpeta data/
if os.path.exists("/content"):
    RUTA_DATOS = "/content/"
    RUTA_RESULTADOS = "/content/"
else:
    RUTA_DATOS = "../data/"
    RUTA_RESULTADOS = "../resultados/"

## 1. Carga de los datos

In [17]:
proyectos = pd.read_excel(RUTA_DATOS + "projects.xlsx")
scivoc = pd.read_excel(RUTA_DATOS + "SciVocCodes.xlsx")

print("Proyectos:", proyectos.shape)
print("Términos EuroSciVoc:", scivoc.shape)

Proyectos: (35378, 19)
Términos EuroSciVoc: (1013, 4)


In [18]:
# La columna euroSciVocCode viene como texto, por ejemplo "[155, 56306972, 325]"
proyectos[["projectID", "title", "euroSciVocCode"]].head()

,projectID,title,euroSciVocCode
0,115843,Ebola Virus: Modern Approaches for developing ...,"[155, 56306972, 325, 137, 1609]"
1,115910,PERtussIS COrrelates of Protection Europe - So...,"[137, 1439, 44109686, 48479582]"
2,635266,Opening the cycling and walking tracking poten...,[1373]
3,637488,"Motivating Motor Learning: The Role of Reward,...","[159, 227, 337, 635]"
4,639226,The Power of Maternal Microbes on Infant Health,"[155, 137, 38005658, 1461, 183]"


In [19]:
# Cada término de la jerarquía tiene su camino completo desde la raíz en full_code
scivoc[["code", "full_code", "full_path"]].head()

,code,full_code,full_path
0,NaN,/27/79/483/,/agricultural sciences/agricultural biotechnol...
1,1275.0,/27/79/483/1275,/agricultural sciences/agricultural biotechnol...
2,483.0,/27/79/483,/agricultural sciences/agricultural biotechnol...
3,481.0,/27/79/481,/agricultural sciences/agricultural biotechnol...
4,487.0,/27/79/487,/agricultural sciences/agricultural biotechnol...


## 2. Construcción de la etiqueta

Cada proyecto tiene una lista de códigos EuroSciVoc. Para cada código, `full_code` nos da el camino
desde la raíz, por ejemplo `/21/35/155`:

- el **primer** número es la categoría de **nivel 1** (21 = *medical and health sciences*),
- el **segundo** número es la categoría de **nivel 2** (35 = *basic medicine*).

Así que "subir por la jerarquía hasta el nivel 2" entendemos que es quedarnos con el segundo elemento del camino.

En el enunciado nos piden que:
1. Se recorren los códigos **en orden** y nos quedamos con el **primero** que tenga nivel 2.
2. Los códigos que no lleguen al nivel 2 los ignoramos.
3. Si ningún código del proyecto vale, el proyecto se descarta.

Además guardamos las otras categorías de nivel 2 del proyecto (**descartadas**), porque nos servirán
en el análisis de errores: si el modelo predice una de ellas, no es un fallo "real".

In [20]:
# Quitamos las 3 filas de SciVocCodes que no tienen código
scivoc = scivoc.dropna(subset=["code"])
scivoc["code"] = scivoc["code"].astype(int)

# Diccionarios para buscar rápido: código -> camino y código -> nombre
camino = dict(zip(scivoc["code"], scivoc["full_code"]))
nombre = dict(zip(scivoc["code"].astype(str), scivoc["title"]))

In [21]:
def obtener_etiquetas(codigos_texto):
    """Devuelve (nivel1, nivel2, descartadas) de un proyecto, o None si no hay etiqueta válida."""
    if pd.isna(codigos_texto):          # proyecto sin códigos
        return None

    codigos = ast.literal_eval(codigos_texto)   # "[155, 325]" -> [155, 325]

    niveles2 = []   # categorías de nivel 2 de todos los códigos, en orden
    nivel1 = None
    for c in codigos:
        if c not in camino:             # código que no está en la jerarquía
            continue
        partes = [x for x in camino[c].split("/") if x != ""]   # "/21/35/155" -> ["21","35","155"]
        if len(partes) < 2:             # no llega al nivel 2
            continue
        if len(niveles2) == 0:          # el primer código válido da la etiqueta
            nivel1 = partes[0]
        niveles2.append(partes[1])

    if len(niveles2) == 0:
        return None

    nivel2 = niveles2[0]
    descartadas = sorted(set(niveles2) - {nivel2})
    return nivel1, nivel2, descartadas

In [22]:
# Comprobamos con el ejemplo del proyecto 115843
ejemplo = proyectos.loc[proyectos["projectID"] == 115843, "euroSciVocCode"].iloc[0]
n1, n2, desc = obtener_etiquetas(ejemplo)
print("Códigos:", ejemplo)
print("Nivel 1:", nombre[n1])
print("Nivel 2:", nombre[n2])
print("Descartadas:", [nombre[d] for d in desc])

Códigos: [155, 56306972, 325, 137, 1609]
Nivel 1: medical and health sciences
Nivel 2: basic medicine
Descartadas: ['health sciences', 'biological sciences']


In [23]:
etiquetas = proyectos["euroSciVocCode"].apply(obtener_etiquetas)

# Nos quedamos solo con los proyectos que tienen etiqueta válida
datos = proyectos[etiquetas.notna()].copy()
etiquetas = etiquetas[etiquetas.notna()]

datos["nivel1"] = [nombre[e[0]] for e in etiquetas]
datos["nivel2"] = [nombre[e[1]] for e in etiquetas]
datos["descartadas"] = [[nombre[d] for d in e[2]] for e in etiquetas]

print(f"Proyectos conservados: {len(datos)} de {len(proyectos)} "
      f"({100 * len(datos) / len(proyectos):.2f} %)  -> el enunciado dice aprox. 90,31 %")
print("Clases de nivel 2:", datos["nivel2"].nunique())
print("Clases de nivel 1:", datos["nivel1"].nunique())

Proyectos conservados: 31982 de 35378 (90.40 %)  -> el enunciado dice aprox. 90,31 %
Clases de nivel 2: 39
Clases de nivel 1: 6


In [24]:
# ¿Cuántos proyectos tienen todos sus códigos en la misma categoría de nivel 2?
inequivocos = (datos["descartadas"].apply(len) == 0).mean()
print(f"Proyectos sin categorías descartadas: {100 * inequivocos:.1f} %  -> el enunciado dice aprox. 32 %")

Proyectos sin categorías descartadas: 32.2 %  -> el enunciado dice aprox. 32 %


In [25]:
# Guardamos el resultado para los siguientes pasos (data/ no se sube a GitHub)
datos.to_pickle(RUTA_DATOS + "proyectos_etiquetados.pkl")

## 3. Partición train / validación / test

Hacemos **una única partición** que usará todo el equipo, para que todos los modelos se comparen
sobre exactamente los mismos proyectos.

- **70 % train**: para entrenar los modelos (y ajustar TF-IDF, Word2Vec, escaladores...).
- **15 % validación**: para elegir hiperparámetros y comparar configuraciones.
- **15 % test**: solo se usa **una vez al final**, para dar el resultado definitivo.

La partición es **estratificada** por la etiqueta de nivel 2: cada conjunto mantiene las mismas
proporciones de clases. Es importante porque las clases están muy desbalanceadas (la más pequeña,
*other medical sciences*, solo tiene 9 proyectos). Fijamos la semilla (`random_state=42`) para que
sea reproducible.

Para el nivel 1 usamos **la misma partición**: los proyectos son los mismos, solo cambia la etiqueta.

In [ ]:
from sklearn.model_selection import train_test_split

# Primero separamos el 15 % de test
train_val, test = train_test_split(datos, test_size=0.15,
                                   stratify=datos["nivel2"], random_state=42)

# Del 85 % restante, separamos validación (0.15 / 0.85 del resto = 15 % del total)
train, val = train_test_split(train_val, test_size=0.15 / 0.85,
                              stratify=train_val["nivel2"], random_state=42)

print(f"Train: {len(train)}  ({100 * len(train) / len(datos):.1f} %)")
print(f"Val:   {len(val)}  ({100 * len(val) / len(datos):.1f} %)")
print(f"Test:  {len(test)}  ({100 * len(test) / len(datos):.1f} %)")

In [ ]:
# Comprobamos que la estratificación funciona: proporción de cada clase en cada conjunto
proporciones = pd.DataFrame({
    "train": train["nivel2"].value_counts(normalize=True),
    "val": val["nivel2"].value_counts(normalize=True),
    "test": test["nivel2"].value_counts(normalize=True),
}).round(3)
proporciones.head(10)

In [ ]:
# Guardamos la partición: un CSV con el identificador de cada proyecto y su conjunto.
# Este fichero SÍ se sube a GitHub: todas lo leen en lugar de hacer su propia partición.
particion = pd.concat([
    pd.DataFrame({"projectID": train["projectID"], "conjunto": "train"}),
    pd.DataFrame({"projectID": val["projectID"], "conjunto": "val"}),
    pd.DataFrame({"projectID": test["projectID"], "conjunto": "test"}),
])
particion.to_csv(RUTA_RESULTADOS + "particion.csv", index=False)
particion["conjunto"].value_counts()